# BERT fine-tuning — CFPB Complaint Triage (Colab GPU)

Fine-tunes `bert-base-uncased` for both tasks (product routing, escalation risk) on a Colab
T4. Self-contained — does **not** `import src`, since Colab will not have the project
package installed. Writes `bert_product.json` and `bert_escalation.json` in the same schema
`src/evaluate.py` already reads, so the files can be copied straight into the local repo's
`outputs/metrics/` directory.

**Before running on Colab:** upload `data/processed/splits.parquet` (or mount Drive and
point at it there) and set `SPLITS_PARQUET_PATH` if it is not at `/content/splits.parquet`.
Set `OUTPUT_DIR` if you want metrics written somewhere other than `/content/outputs`.

**Running locally instead** (e.g. to sanity-check the notebook before uploading it): launch
this notebook from the `notebooks/` directory or the repo root with the `nlp-final` kernel
and it will find `data/processed/splits.parquet` on its own — no env var needed. Training
will be slow without a GPU; use `QUICK_MODE = True` below for a local smoke test.

**Runtime (Colab):** Runtime → Change runtime type → T4 GPU, before running the cells
below.

In [1]:
!pip install -q "transformers>=4.40" scikit-learn pandas pyarrow

In [2]:
import os
from pathlib import Path

_env_path = os.environ.get("SPLITS_PARQUET_PATH")
_candidates = [_env_path] if _env_path else []
_candidates += [
    "/content/splits.parquet",                # Colab, after uploading the file
    "../data/processed/splits.parquet",        # running locally from notebooks/
    "data/processed/splits.parquet",           # running locally from the repo root
]
_found = next((p for p in _candidates if p and Path(p).exists()), None)
if _found is None:
    raise FileNotFoundError(
        "splits.parquet not found. On Colab: upload data/processed/splits.parquet to "
        "/content/ (or mount Drive and set SPLITS_PARQUET_PATH). Running locally: run this "
        "notebook from the notebooks/ directory or the repo root, or set "
        "SPLITS_PARQUET_PATH explicitly. Checked: " + ", ".join(c for c in _candidates if c)
    )
SPLITS_PARQUET_PATH = _found
print(f"using splits.parquet at: {SPLITS_PARQUET_PATH}")

OUTPUT_DIR = os.environ.get("OUTPUT_DIR", "/content/outputs")

SEED = 42  # must match src/config.py SEED - do not change independently of the sklearn runs

using splits.parquet at: ../data/processed/splits.parquet


In [3]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    average_precision_score, roc_auc_score,
)

torch.manual_seed(SEED)

METRICS_DIR = Path(OUTPUT_DIR) / "metrics"
METRICS_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")
if device.type != "cuda":
    print("WARNING: no GPU detected. Runtime > Change runtime type > T4 GPU, then re-run.")

device: cpu


## Load the frozen splits

Reads the same `splits.parquet` the sklearn baselines use. Never resplit — the train/val/test
assignment must match the local run exactly or the comparison across models is worthless.

In [4]:
def load_split(name, path=SPLITS_PARQUET_PATH):
    df = pd.read_parquet(path)
    return df[df["split"] == name].reset_index(drop=True)

In [ ]:
QUICK = {
    "mode": "quick",
    "model_name": "distilbert-base-uncased",
    "max_length": 128,
    "epochs": 1,
    "train_docs": 2000,
    "test_docs": 1000,
    "batch_size": 16,
}

FULL = {
    "mode": "full",
    "model_name": "bert-base-uncased",
    "max_length": 256,
    "epochs": 2,
    "train_docs": None,
    "test_docs": None,
    "batch_size": 16,
}

# True for a fast sanity check on a CPU or small-GPU Colab runtime; False (default) is the
QUICK_MODE = False
cfg = QUICK if QUICK_MODE else FULL
cfg

{'mode': 'full',
 'model_name': 'bert-base-uncased',
 'max_length': 256,
 'epochs': 2,
 'train_docs': None,
 'test_docs': None,
 'batch_size': 16}

In [7]:
class NarrativeDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], truncation=True, max_length=self.max_length,
            padding="max_length", return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

In [8]:
def prepare_labels(task_name, tr, te):
    if task_name == "product":
        classes = sorted(tr["product"].unique())
        class_to_idx = {c: i for i, c in enumerate(classes)}
        y_tr = tr["product"].map(class_to_idx).to_numpy()
        y_te = te["product"].map(class_to_idx).to_numpy()
    else:
        classes = [0, 1]
        class_to_idx = {0: 0, 1: 1}
        y_tr = tr["escalated"].to_numpy()
        y_te = te["escalated"].to_numpy()
    return classes, class_to_idx, y_tr, y_te


def encode_labels(task_name, df, class_to_idx):
    col = "product" if task_name == "product" else "escalated"
    return df[col].map(class_to_idx).to_numpy()


def run_eval(model, loader, device):
    model.eval()
    all_preds, all_probs = [], []
    forward_seconds = 0.0
    t0 = time.time()
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            batch.pop("labels")
            tf0 = time.time()
            outputs = model(**batch)
            forward_seconds += time.time() - tf0
            probs = torch.softmax(outputs.logits, dim=-1)
            all_probs.append(probs.cpu().numpy())
            all_preds.append(probs.argmax(dim=-1).cpu().numpy())
    end_to_end_seconds = time.time() - t0
    return np.concatenate(all_preds), np.concatenate(all_probs), end_to_end_seconds, forward_seconds


def decode_predictions(task_name, preds_idx, idx_to_class):
    if task_name == "product":
        return np.array([idx_to_class[i] for i in preds_idx], dtype=object)
    return preds_idx.astype(int)

## Training loop

Manual PyTorch loop, not the `Trainer` API — same reasoning as `src/transformer.py`: the
`Trainer` surface changes between `transformers` versions and that is not worth debugging
under deadline. Mixed precision only activates when CUDA is available. Class weights are
computed from the (possibly subsampled) training set actually used, not the full split, and
handed to `CrossEntropyLoss` — printed explicitly below so a collapsed positive-class score
can be told apart from a dropped-weights bug.

In [9]:
def run_task(task_name, cfg):
    torch.manual_seed(SEED)
    use_amp = device.type == "cuda"

    tr = load_split("train")
    va = load_split("val")
    te = load_split("test")
    if cfg["train_docs"]:
        tr = tr.sample(n=min(cfg["train_docs"], len(tr)), random_state=SEED).reset_index(drop=True)
    if cfg["test_docs"]:
        va = va.sample(n=min(cfg["test_docs"], len(va)), random_state=SEED).reset_index(drop=True)
        te = te.sample(n=min(cfg["test_docs"], len(te)), random_state=SEED).reset_index(drop=True)

    classes, class_to_idx, y_tr, y_te = prepare_labels(task_name, tr, te)
    y_va = encode_labels(task_name, va, class_to_idx)
    num_labels = len(classes)
    idx_to_class = {i: c for c, i in class_to_idx.items()}

    print(f"\n=== {task_name} | {cfg['mode']} | model={cfg['model_name']} | "
          f"train={len(tr)} val={len(va)} test={len(te)} device={device} ===")

    tokenizer = AutoTokenizer.from_pretrained(cfg["model_name"])
    model = AutoModelForSequenceClassification.from_pretrained(
        cfg["model_name"], num_labels=num_labels
    ).to(device)

    train_loader = DataLoader(
        NarrativeDataset(tr.narrative, y_tr, tokenizer, cfg["max_length"]),
        batch_size=cfg["batch_size"], shuffle=True,
    )
    val_loader = DataLoader(
        NarrativeDataset(va.narrative, y_va, tokenizer, cfg["max_length"]),
        batch_size=cfg["batch_size"], shuffle=False,
    )
    test_loader = DataLoader(
        NarrativeDataset(te.narrative, y_te, tokenizer, cfg["max_length"]),
        batch_size=cfg["batch_size"], shuffle=False,
    )

    class_weights = None
    if task_name == "escalation":
        n_pos = max(int((y_tr == 1).sum()), 1)
        n_neg = int((y_tr == 0).sum())
        class_weights = torch.tensor([1.0, n_neg / n_pos], dtype=torch.float, device=device)
        print(f"class weights: n_train_docs={len(tr)} n_pos={n_pos} n_neg={n_neg} "
              f"weight_tensor={class_weights.tolist()}")

    optimizer = AdamW(model.parameters(), lr=2e-5)
    loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)
    if class_weights is not None:
        matches = torch.equal(loss_fn.weight.cpu(), class_weights.cpu())
        print(f"loss_fn.weight={loss_fn.weight.tolist()} matches computed tensor: {matches}")
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)

    model.train()
    t0 = time.time()
    for epoch in range(cfg["epochs"]):
        running_loss, n_steps = 0.0, 0
        for step, batch in enumerate(train_loader):
            batch = {k: v.to(device) for k, v in batch.items()}
            labels = batch.pop("labels")
            optimizer.zero_grad()
            with torch.autocast(device_type=device.type, enabled=use_amp):
                outputs = model(**batch)
                loss = loss_fn(outputs.logits, labels)
            if use_amp:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()
            running_loss += loss.item()
            n_steps += 1
            if step % 50 == 0:
                print(f"epoch {epoch + 1} step {step}/{len(train_loader)} loss={loss.item():.4f}")
        print(f"epoch {epoch + 1} mean loss={running_loss / n_steps:.4f}")
        # If the loss has stopped dropping after this epoch, do not add epochs - BERT
        # converges fast on this data and a third epoch tends to just overfit.
    train_seconds = time.time() - t0

    preds, probs, end_to_end_seconds, forward_seconds = run_eval(model, test_loader, device)
    val_preds, val_probs, _, _ = run_eval(model, val_loader, device)
    n_test = len(te)

    np.save(Path(OUTPUT_DIR) / f"{task_name}_bert_pred.npy", decode_predictions(task_name, preds, idx_to_class))
    np.save(Path(OUTPUT_DIR) / f"{task_name}_bert_val_pred.npy", decode_predictions(task_name, val_preds, idx_to_class))
    if task_name == "escalation":
        np.save(Path(OUTPUT_DIR) / f"{task_name}_bert_proba.npy", probs[:, 1])
        np.save(Path(OUTPUT_DIR) / f"{task_name}_bert_val_proba.npy", val_probs[:, 1])

    accuracy = accuracy_score(y_te, preds)
    macro_f1 = f1_score(y_te, preds, average="macro")
    target_names = [str(idx_to_class[i]) for i in range(num_labels)]
    report = classification_report(y_te, preds, target_names=target_names, output_dict=True, zero_division=0)

    result = {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "train_seconds": round(train_seconds, 2),
        "end_to_end_seconds": round(end_to_end_seconds, 3),
        "predict_seconds": round(forward_seconds, 3),
        "inference_n_docs": n_test,
        "train_n_docs": len(tr),
        "report": report,
        "mode": cfg["mode"],
        "model_name": cfg["model_name"],
        "epochs": cfg["epochs"],
        "max_length": cfg["max_length"],
    }
    if task_name == "escalation":
        pos_probs = probs[:, 1]
        result["pr_auc"] = average_precision_score(y_te, pos_probs)
        result["roc_auc"] = roc_auc_score(y_te, pos_probs)
        result["positive_f1"] = f1_score(y_te, preds, pos_label=1, zero_division=0)

    docs_per_sec = n_test / end_to_end_seconds if end_to_end_seconds else float("nan")
    print(f"{task_name:12s} acc={accuracy:.4f}  macroF1={macro_f1:.4f}  "
          f"train={train_seconds:.1f}s  end-to-end={docs_per_sec:.1f} docs/sec"
          + (f"  PR-AUC={result['pr_auc']:.4f}" if "pr_auc" in result else ""))

    out_path = METRICS_DIR / f"bert_{task_name}.json"
    with open(out_path, "w") as f:
        json.dump({"bert": result}, f, indent=2)
    print(f"wrote {out_path}")
    return result

## Run both tasks

In [ ]:
product_result = run_task("product", cfg)
escalation_result = run_task("escalation", cfg)


=== product | full | model=bert-base-uncased | train=37800 test=8100 device=cpu ===


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

c:\Users\maina\anaconda3\envs\nlp-final\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\maina\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


epoch 1 step 0/2363 loss=2.1448


## Copy results back to the project

Download `bert_product.json` and `bert_escalation.json` from `OUTPUT_DIR/metrics/` (zipped
below for convenience) and place them in the local repo's `outputs/metrics/` directory, then
run `python -m src.evaluate` locally to fold them into the comparison tables.

In [ ]:
import shutil

archive_path = shutil.make_archive("/content/bert_outputs", "zip", OUTPUT_DIR)
print(f"zipped to {archive_path} - download it via the Colab file browser")
print("contains: metrics/bert_product.json, metrics/bert_escalation.json, and the "
      "*_bert_pred.npy / *_bert_val_pred.npy / *_bert_proba.npy / *_bert_val_proba.npy "
      "prediction arrays")